# Import modules

In [1]:
import os
import sys
import requests                      # HTTP client for API calls
import pandas as pd                  # Tabular data handling
from datetime import datetime        # Datetime handling
from typing import Iterable, Optional, Dict, Union
import matplotlib.pyplot as plt
import yfinance as yf
from pprint import pprint as pp

Ensure repo root is on PYTHONPATH (CI safety)

In [2]:
REPO_ROOT = os.path.abspath(os.getcwd())
SRC_PATH = os.path.join(REPO_ROOT, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

Secrets validation (FAIL FAST)

In [3]:
from src.debug_print import debug_print

try:
    REQUIRED_ENV_VARS = [
        "EMAIL_USER",
        "EMAIL_SENDER",
        "EMAIL_SENDER_PSW",
    ]

    missing = [v for v in REQUIRED_ENV_VARS if not os.getenv(v)]
    if missing:
        raise RuntimeError(
            f"Missing required environment variables: {', '.join(missing)}"
        )

    email_user = os.getenv("EMAIL_USER")
    email_sender_psw = os.getenv("EMAIL_SENDER_PSW")
    email_sender = os.getenv("EMAIL_SENDER")
except Exception as e:
    print(f"{debug_print()}\n Could not find env vars for email users\n{type(e).__name__}: {e}")



Import internal modules

In [4]:
from src.fetch_lse_tickers import get_ftse100
from src.exchange_rates_v2 import get_share_prices_2_with_fundamentals
from src.plot_shares_ROI import plot_candles_volatility_volume_roi as ROI
from src.extract_latest_fundamentals import extract_latest_fundamentals
from src.detect_undervalued import detect_undervalued
from src.utils.email_sender import send_email_html_multi_inline_images 
from src.utils.email_undervalued_shares import build_undervalued_shares_email_with_images
from src.purchase_price import get_purchase_price
from src.utils.build_portfolio_email_html import build_roi_email_from_df


Check Picture and portfolio folders exists 

In [ ]:

pics_dir = os.path.join(os.getcwd(), "output")
os.makedirs(pics_dir, exist_ok=True)
os.makedirs("PORTFOLIO", exist_ok=True)
portfolio_path = os.path.join(os.getcwd(), "PORTFOLIO", "purchases.csv")

# Set up variables

In [6]:
base_currency = "GBP"

start_date = pd.Timestamp(2025,1,1)
end_date = pd.Timestamp.today().normalize()+pd.Timedelta(days=1)
ROI_target = 0.135

email_recipients = ["ingcarldan@gmail.com"]
email_user = os.getenv("EMAIL_USER")
email_sender_psw = os.getenv("EMAIL_SENDER_PSW")
email_sender = os.getenv("EMAIL_SENDER")

# portfolio setup
new_shares = False
old_portfolio_setup = False
pur = pd.read_csv(portfolio_path, parse_dates=['Purchase_Date'])
pur = pur.drop(columns=['Unnamed: 0'], errors='ignore')

if new_shares == True:
    new_row = {
        "Action": "TEMP_New_action", 
        "Purchase_Date": pd.Timestamp.today().normalize(),
        "Purchase_Price": 0.00,
        "Currency": "GBP",
        "Target_ROI":0.135
        }
    pur = pd.concat([pur, pd.DataFrame([new_row])], ignore_index=True)
    if old_portfolio_setup == True:
        purchase_dates = {
            'FRES': pd.Timestamp(2026,1,3),
            'NWG':pd.Timestamp(2026,1,7),
            'GLEN':pd.Timestamp(2026,1,6),
            'RIO':pd.Timestamp(2026,1,8),
        }
        pur = pd.DataFrame(purchase_dates.items(), columns=['Action', 'Purchase_Date'])
        pur[['Purchase_price', 'Current_price' , 'Currency', 'Current_ROI', 'Target_ROI', 'ROI_reached']] = None
        pur['Currency'] = pur['Currency'].astype('string')
        pur['ROI_reached'] = pur['ROI_reached'].astype('bool')
        pur[['Current_ROI', 'Target_ROI']] = pur[['Current_ROI', 'Target_ROI']].astype('float')

        # pp(pur)
        input_cols = ['Purchase_price', 'Currency', 'Target_ROI']
        pur.loc[0, input_cols] = [34.78, 'GBP', 0.135] # Purchase_price FRES
        pur.loc[1, input_cols] = [6.35, 'GBP', 0.135] # Purchase_price NWG
        pur.loc[2, input_cols] = [4.2495, 'GBP', 0.135] # Purchase_price GLEN
        pur.loc[3, input_cols] = [62, 'GBP', 0.135] # Purchase_price RIO
        pur.loc[:, 'ROI_reached'] = False # set default to False

        print(pur)


# Get TOP 100 shares from FTSE

In [7]:
ftse100 = get_ftse100()
ftse100["Yahoo_Ticker"] = ftse100["Ticker"] + ".L"
shares_lse = ftse100["Yahoo_Ticker"].to_list()

[DEBUG PRINT]
   File: c:\Users\ingca\OneDrive\Documents\python\finance\src\utils\retry_decorator.py
   Function: wrapper
   Line: 38
None get_ftse100 succeeded


# SHARES PRICES WITH INFO

In [8]:
df_shares_fund, failed_tickers_list = get_share_prices_2_with_fundamentals(
    tickers=shares_lse,# list with items ending with ".L"
    start=start_date,
    end=end_date,
    base_currency = base_currency,
    vol_window = 20,
    
)

actions_list   = df_shares_fund.columns.get_level_values("ACTION").unique().to_list()
currencies_list = df_shares_fund.columns.get_level_values("CURRENCY").unique()
metrics   = df_shares_fund.columns.get_level_values("METRIC").unique()

# GET PURCHASE PRICES IF NOT PROVIDED

In [9]:
# Get purchase price if not provided using timestamp as reference
# if timestamp not found in df, revert to previous available date

for action, purchase_date in pur[['Action', 'Purchase_Date']].itertuples(index=False):

    idx = pur.index[
        (pur['Action'] == action) &
        (pur['Purchase_Date'] == purchase_date)
    ][0]
    action_full = next(act for act in actions_list if act.startswith(action))
    currency = action_full[-3:]

    # print(f"Idx for action {action} and purchase date {purchase_date} is {idx}")

    if pd.isna(pur.loc[idx, 'Purchase_price']):
       
        HIGH = get_purchase_price(
            df=df_shares_fund,
            action=action_full,
            currency=currency,
            metric="HIGH",
            date=purchase_date          # <-- scalar Timestamp
        )
        LOW = get_purchase_price(
            df=df_shares_fund,
            action=action_full,
            currency=currency,
            metric="LOW",
            date=purchase_date          # <-- scalar Timestamp
        )
        pur.loc[idx, 'Purchase_price'] = (HIGH + LOW) / 2

    current_value = df_shares_fund[(action_full,action_full[-3:],'CLOSE')].iloc[-1]
    pur.loc[idx, 'Current_price'] = current_value
    pur.loc[idx, 'Currency'] = currency
    pur.loc[idx, 'Current_ROI'] = round((current_value - pur.loc[idx, 'Purchase_price']) / pur.loc[idx, 'Purchase_price'],4)
    pur.loc[idx, 'ROI_reached'] = pur.loc[idx, 'Current_ROI'] >= pur.loc[idx, 'Target_ROI']

# Save updated portfolio
pur.to_csv(portfolio_path, index=False)

# EXTRACT UNDERVALUED SHARES

In [10]:
df_fund = extract_latest_fundamentals(
    df=df_shares_fund,
    evaluation_date=end_date,
)

undervalued_shares = detect_undervalued(df_fund)
filt = undervalued_shares[undervalued_shares["UndervaluedScore"] > 0]
undervalued_shares_list = filt.index.to_list()
filt["Ticker"] = filt.index.str.split(".").str[0]

filt["Company"] = (
    filt["Ticker"]
    .map(ftse100.set_index("Ticker")["Company"])
)
currency_convertion = [s.split('_')[1] for s in filt.index.to_list()]

filt['original currency'] = [s.split('→')[0] for s in currency_convertion]
filt['converted currency'] = [s.split('→')[1] for s in currency_convertion]
filt = filt[['Company', 'Ticker', 'UndervaluedScore', 'original currency', 'converted currency', 
                'Price', 'EPS', 'BookValue', 'Dividend', 'P/E', 'P/B']]
ROI( # from src.plot_shares_ROI import plot_candles_volatility_volume_roi
    df=df_shares_fund,
    actions=filt.index.to_list(), # list with items ending with ".L_GBp→GBP"
    start=df_shares_fund.index.min(),
    end=df_shares_fund.index.max(),
    roi_target=ROI_target,
    plot_purchase=False
)

C:\Users\ingca\AppData\Local\Temp\ipykernel_2896\3826778731.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filt["Ticker"] = filt.index.str.split(".").str[0]
C:\Users\ingca\AppData\Local\Temp\ipykernel_2896\3826778731.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filt["Company"] = (
C:\Users\ingca\AppData\Local\Temp\ipykernel_2896\3826778731.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

S

[DEBUG PRINT]
   File: c:\Users\ingca\OneDrive\Documents\python\finance\src\utils\retry_decorator.py
   Function: wrapper
   Line: 38
None plot_candles_volatility_volume_roi succeeded


# SEND EMAIL FOR UNDERVALUED SHARES

In [11]:
try:
    text_body, html_body, inline_images = build_undervalued_shares_email_with_images(
        df_undervalued_shares=filt,
        image_dir='output'
    )

except Exception as e:
    print(f"ERROR in build_undervalued_shares_email_with_images {type(e).__name__}: {e}")
try:
    send_email_html_multi_inline_images(
        smtp_server="smtp.gmail.com",
        smtp_port=587,
        username=email_user,
        password=email_sender_psw,
        sender=email_sender,
        recipients=email_recipients,
        subject=f"FTSE100 - Undevalued shares – {datetime.today():%d-%m-%Y %H:%M}",
        text_body=text_body,
        html_body=html_body,
        inline_images=inline_images,
    )
except Exception as e:
    print(f"Could not run send_email_html_multi_inline_images {type(e).__name__}: {e}")
        


[DEBUG PRINT]
   File: c:\Users\ingca\OneDrive\Documents\python\finance\src\utils\email_undervalued_shares.py
   Function: build_undervalued_shares_email_with_images
   Line: 64
None image_dir: c:\Users\ingca\OneDrive\Documents\python\finance\output
[DEBUG PRINT]
   File: c:\Users\ingca\OneDrive\Documents\python\finance\src\utils\retry_decorator.py
   Function: wrapper
   Line: 38
None send_email_html_multi_inline_images succeeded


portfolio = {}
SHARES_FULL_LIST = [s + '.L_GBp→GBP' if not s.endswith('.L_GBp→GBP') else s for s in list(purchase_dates.keys())]

for action in SHARES_FULL_LIST:
    try:
        action_clean = action.split(".L")[0]
        portfolio[action_clean] = get_first_roi_hit(
        df=df_shares_fund,
        action=action,
        purchase_dates=purchase_dates,
        roi_target=ROI_target
    )
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

#  PLOT SHARE - plot_candles_volatility_volume_roi 


In [12]:
filtered_actions = [s for s in actions_list if s.startswith(tuple(pur['Action'].to_list()))]

# save input for DEBUG
debug_file_option = False
if debug_file_option == True:
    import pickle
    df_shares_fund.to_pickle("df_shares_fund.pkl")
    pur.to_pickle("pur.pkl")

    with open("filtered_actions.pkl", "wb") as f:
        pickle.dump(filtered_actions, f)

    with open("ROI_target.pkl", "wb") as f:
        pickle.dump(ROI_target, f)



ROI( # from src.plot_shares_ROI import plot_candles_volatility_volume_roi
    df=df_shares_fund,
    actions=filtered_actions,
    start=df_shares_fund.index.min(),
    end=df_shares_fund.index.max(),
    purchase_dates=pur,
    roi_target=ROI_target,
)

[DEBUG PRINT]
   File: c:\Users\ingca\OneDrive\Documents\python\finance\src\utils\retry_decorator.py
   Function: wrapper
   Line: 38
None plot_candles_volatility_volume_roi succeeded


# ROI email HTML body setup

In [13]:
image_dir = os.path.join(os.getcwd(), "output")
try:
    roi_text_body, roi_html_body, roi_inline_images = build_roi_email_from_df(
        df=pur,
        action_list = actions_list,
        image_dir = image_dir,
    )
    # print(f"actions_list:\n {actions_list}")
    # print(f"text_body: {text_body}\nhtml_body: {html_body}\ninline_images keys: {list(inline_images.keys())}\n")
except Exception as e:
    print(f"Could not run build_roi_email_from_df {type(e).__name__}: {e}")


DEBUG CELL

print("roi_text_body")
pp(roi_text_body)
print("roi_html_body")
pp(roi_html_body)
print("roi_inline_images")
pp(roi_inline_images)

# Send email (TLS via your utility)

In [14]:

try:
    send_email_html_multi_inline_images(
        smtp_server="smtp.gmail.com",
        smtp_port=587,
        username=email_user,
        password=email_sender_psw,
        sender=email_sender,
        recipients=email_recipients,
        subject=f"FTSE100 - ROI targets – {datetime.today():%d-%m-%Y %H:%M}",
        text_body=roi_text_body,
        html_body=roi_html_body,
        inline_images=roi_inline_images,
    )
except Exception as e:
    print(f"Could not run send_email_html_multi_inline_images {type(e).__name__}: {e}")

[DEBUG PRINT]
   File: c:\Users\ingca\OneDrive\Documents\python\finance\src\utils\retry_decorator.py
   Function: wrapper
   Line: 38
None send_email_html_multi_inline_images succeeded
